In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from pathlib import Path
from nilearn.maskers import NiftiMasker

from abstract_values.utils.data import Subject, BIDS_FOLDER

In [ ]:
subject  = 'pil01'
sessions = [1, 2]

sub = Subject(subject, bids_folder=BIDS_FOLDER)
deriv = BIDS_FOLDER / 'derivatives' / 'encoding_models'

## Load mean CV R² images

Three models:
- **vonmises** — Von Mises basis set, fitted to Gabor *orientation* (should win in V1)
- **aprf** — Log-Gaussian pRF, fitted to stimulus *value* (should win in NPCr/IPS)
- **aprf-shift** — Log-Gaussian pRF with session-shifted mode, fitted to *value* (extra flexibility)

In [ ]:
def load_mean_cvr2(model_dir_name):
    """Load mean CV R² NIfTI for subject (both sessions → no ses entity)."""
    fn = (deriv / model_dir_name / f'sub-{subject}' / 'func'
          / f'sub-{subject}_task-abstractvalue_space-T1w_desc-cvr2_pe.nii.gz')
    if not fn.exists():
        raise FileNotFoundError(fn)
    return nib.load(str(fn))

imgs = {
    'vonmises':  load_mean_cvr2('vonmises.cv'),
    'aprf':      load_mean_cvr2('aprf.cv'),
    'aprf-shift': load_mean_cvr2('aprf-shift.cv'),
}

for name, img in imgs.items():
    print(f'{name:12s}  shape={img.shape}')

## ROI masks

- **V1**: bilateral Benson V1 (`hemi-LR`)
- **NPCr / IPS**: right Numerosity Parietal Cortex (`hemi=None`)

In [ ]:
rois = {
    'V1 (bilateral)': sub.get_roi_mask('BensonV1', hemi='LR'),
    'NPCr (IPS)':     sub.get_roi_mask('NPCr', hemi=None),
}

for name, mask in rois.items():
    n = int(nib.load(str(mask)).get_fdata().sum()) if isinstance(mask, (str, Path)) else int(mask.get_fdata().sum())
    print(f'{name:20s}  {n:,} voxels')

## Extract CV R² per ROI

In [ ]:
# {roi_name: {model_name: array of CV R² values}}
roi_data = {}

for roi_name, mask in rois.items():
    masker = NiftiMasker(mask_img=mask).fit()
    roi_data[roi_name] = {
        model: masker.transform(img).squeeze()
        for model, img in imgs.items()
    }
    n = len(next(iter(roi_data[roi_name].values())))
    print(f'{roi_name}: {n:,} voxels extracted')

## Mean CV R² per model and ROI

In [ ]:
model_labels = {
    'vonmises':   'VonMises\n(orientation)',
    'aprf':       'aPRF\n(value)',
    'aprf-shift': 'aPRF-shift\n(value+session)',
}
model_colors = {
    'vonmises':   '#4c72b0',
    'aprf':       '#dd8452',
    'aprf-shift': '#55a868',
}

fig, axes = plt.subplots(1, len(rois), figsize=(5 * len(rois), 5), sharey=False)

for ax, (roi_name, model_vals) in zip(axes, roi_data.items()):
    means  = [model_vals[m].mean() for m in model_labels]
    sems   = [model_vals[m].std() / np.sqrt(len(model_vals[m])) for m in model_labels]
    labels = [model_labels[m] for m in model_labels]
    colors = [model_colors[m] for m in model_labels]

    bars = ax.bar(range(len(means)), means, yerr=sems, capsize=4,
                  color=colors, edgecolor='k', linewidth=0.8)
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('Mean CV R²', fontsize=11)
    ax.set_title(f'sub-{subject}  {roi_name}', fontsize=11)
    ax.axhline(0, color='k', lw=0.8)

plt.tight_layout()
plt.show()

## CV R² distributions (violin plots)

In [ ]:
fig, axes = plt.subplots(1, len(rois), figsize=(5 * len(rois), 5), sharey=False)

for ax, (roi_name, model_vals) in zip(axes, roi_data.items()):
    data_list  = [model_vals[m] for m in model_labels]
    labels     = [model_labels[m] for m in model_labels]
    colors     = [model_colors[m] for m in model_labels]

    parts = ax.violinplot(data_list, positions=range(len(data_list)),
                          showmedians=True, showextrema=False)
    for body, color in zip(parts['bodies'], colors):
        body.set_facecolor(color)
        body.set_alpha(0.7)
    parts['cmedians'].set_color('k')

    ax.set_xticks(range(len(data_list)))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('CV R²', fontsize=11)
    ax.set_title(f'sub-{subject}  {roi_name}', fontsize=11)
    ax.axhline(0, color='k', lw=0.8, linestyle='--')

plt.tight_layout()
plt.show()

## Winner map: which model has highest CV R² per voxel?

Winner is assigned only where at least one model exceeds a positive threshold.

In [ ]:
cv_r2_thr = 0.02   # minimum CV R² to call a voxel 'tuned'

model_names = list(model_labels.keys())

for roi_name, model_vals in roi_data.items():
    stack = np.stack([model_vals[m] for m in model_names], axis=1)  # (n_vox, 3)
    max_cvr2 = stack.max(axis=1)
    winner   = np.argmax(stack, axis=1)  # 0=vonmises, 1=aprf, 2=aprf-shift
    tuned    = max_cvr2 >= cv_r2_thr

    print(f'\n{roi_name}  (CV R² ≥ {cv_r2_thr}, n={tuned.sum():,}/{len(tuned):,} = {100*tuned.mean():.1f}%)')
    for i, m in enumerate(model_names):
        frac = (winner[tuned] == i).mean()
        n    = (winner[tuned] == i).sum()
        print(f'  {model_labels[m]:28s}  {frac*100:5.1f}%  ({n:,} voxels)')

In [ ]:
# Bar chart: fraction of tuned voxels won by each model, per ROI
cv_r2_thr = 0.02

fig, axes = plt.subplots(1, len(rois), figsize=(5 * len(rois), 5))

for ax, (roi_name, model_vals) in zip(axes, roi_data.items()):
    stack  = np.stack([model_vals[m] for m in model_names], axis=1)
    winner = np.argmax(stack, axis=1)
    tuned  = stack.max(axis=1) >= cv_r2_thr

    fracs  = [(winner[tuned] == i).mean() for i in range(len(model_names))]
    labels = [model_labels[m] for m in model_names]
    colors = [model_colors[m] for m in model_names]

    bars = ax.bar(range(len(fracs)), [f * 100 for f in fracs],
                  color=colors, edgecolor='k', linewidth=0.8)
    ax.set_xticks(range(len(fracs)))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('% of tuned voxels won', fontsize=11)
    ax.set_title(f'sub-{subject}  {roi_name}\n(CV R² ≥ {cv_r2_thr}, n={tuned.sum():,})',
                 fontsize=11)
    ax.set_ylim(0, 100)

plt.suptitle('Model winner comparison (leave-one-run-out CV)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Pairwise ΔCV R²: value model vs orientation model

Positive = value model better; negative = orientation model better.

In [ ]:
fig, axes = plt.subplots(1, len(rois), figsize=(5 * len(rois), 4))

for ax, (roi_name, model_vals) in zip(axes, roi_data.items()):
    delta_aprf       = model_vals['aprf']       - model_vals['vonmises']
    delta_aprf_shift = model_vals['aprf-shift'] - model_vals['vonmises']

    ax.hist(delta_aprf,       bins=60, alpha=0.6, color=model_colors['aprf'],
            label=f'aPRF − VonMises  (mean={delta_aprf.mean():.3f})')
    ax.hist(delta_aprf_shift, bins=60, alpha=0.6, color=model_colors['aprf-shift'],
            label=f'aPRF-shift − VonMises  (mean={delta_aprf_shift.mean():.3f})')
    ax.axvline(0, color='k', lw=1.2, linestyle='--')
    ax.set_xlabel('ΔCV R² (value model − orientation model)', fontsize=10)
    ax.set_ylabel('voxel count', fontsize=10)
    ax.set_title(f'sub-{subject}  {roi_name}', fontsize=11)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()